# Appliance Energy Forecasting — Part 5 & 6: Covariates & Feature-Based ML Model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/appliance-energy-forecasting"
os.chdir(PROJECT_ROOT)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

plt.rcParams["figure.figsize"] = (14, 5)
np.random.seed(0)

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/forecasts", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)


In [ ]:
hourly = pd.read_csv("data/processed/appliance_hourly.csv", index_col=0, parse_dates=True)

TARGET = "Appliances"
HORIZON = 24
DAILY_PERIOD = 24
TEST_STEPS = 14 * 24

print(hourly.shape)
hourly.head()


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mase(y_true, y_pred, y_train, seasonality=24):
    y_train = pd.Series(y_train).astype(float)
    seasonal_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = seasonal_errors.mean()
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

def evaluate_forecast(name, y_true, y_pred, y_train):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD),
        "Bias": np.mean(y_pred - y_true),
    }


### Time-based features

hour/day-of-week are known at any forecast origin, so are safe to use without any risk
of leakage.

In [ ]:
def add_time_features(df):
    out = df.copy()
    out["hour"] = out.index.hour
    out["dayofweek"] = out.index.dayofweek
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["dayofweek"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dayofweek"] / 7)
    return out


### Lag and rolling features

All lag/rolling features use `.shift(1)` before rolling, so the rolling window never
includes the current or future target value. This avoids leakage into the model.

In [ ]:
def make_ml_table(df, target=TARGET):
    out = add_time_features(df)

    lags = [1, 2, 3, 6, 12, 24, 48, 168]
    for lag in lags:
        out[f"lag_{lag}"] = out[target].shift(lag)

    windows = [3, 6, 12, 24, 168]
    for window in windows:
        out[f"roll_mean_{window}"] = out[target].shift(1).rolling(window).mean()
        out[f"roll_std_{window}"] = out[target].shift(1).rolling(window).std()

    return out.dropna()

ml_data = make_ml_table(hourly, target=TARGET)
print(ml_data.shape)
ml_data.head()


In [ ]:
ml_train = ml_data.iloc[:-TEST_STEPS]
ml_test = ml_data.iloc[-TEST_STEPS:]

feature_cols = [c for c in ml_data.columns if c != TARGET]

X_train_ml = ml_train[feature_cols]
y_train_ml = ml_train[TARGET]
X_test_ml = ml_test[feature_cols]
y_test_ml = ml_test[TARGET]

print("features used:", len(feature_cols))
print(feature_cols)


In [ ]:
feature_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.03,
    max_leaf_nodes=31,
    random_state=0,
)
feature_model.fit(X_train_ml, y_train_ml)

feature_pred = pd.Series(feature_model.predict(X_test_ml), index=y_test_ml.index, name="feature_model")


In [ ]:
feature_results = evaluate_forecast("feature_model", y_test_ml, feature_pred, y_train_ml)
print(feature_results)


### Feature importance

Permutation importance shows which feature groups (lags, rolling stats, time features,
sensor/weather readings) actually contribute to prediction accuracy.

In [ ]:
perm_result = permutation_importance(
    feature_model, X_test_ml, y_test_ml, n_repeats=5, random_state=0, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)

print(importance_df.head(15))

fig, ax = plt.subplots(figsize=(10, 8))
top_features = importance_df.head(15)
ax.barh(top_features["feature"], top_features["importance_mean"])
ax.invert_yaxis()
ax.set_title("Top 15 Feature Importances (Permutation)")
ax.set_xlabel("Mean decrease in score")
plt.tight_layout()
plt.savefig("outputs/figures/09_feature_importance.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_window = y_test_ml.index[:24*7]

y_test_ml.loc[plot_window].plot(ax=ax, label="actual", color="black", linewidth=2)
feature_pred.reindex(plot_window).plot(ax=ax, label="feature_model forecast", color="tab:green")
ax.set_title("Feature-Based Model Forecast vs Actual - First 7 Days of Test Period")
ax.set_ylabel("Appliances (Wh)")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/10_feature_model_forecast.png", dpi=150)
plt.show()


In [ ]:
feature_forecast_df = pd.DataFrame({
    "actual": y_test_ml,
    "feature_model": feature_pred,
})
feature_forecast_df.to_csv("outputs/forecasts/feature_model_forecast.csv")

pd.DataFrame([feature_results]).to_csv("outputs/metrics/feature_model_metrics.csv", index=False)
importance_df.to_csv("outputs/metrics/feature_importance.csv", index=False)
print("saved feature model forecast, metrics, and importance")
